# **02_eda_analisis_demanda**

Este notebook desarrolla el análisis exploratorio de la serie de demanda eléctrica diaria
a partir del dataset procesado `data_daily_ready.csv`.

## Objetivos
- Describir estadísticamente la serie objetivo.
- Visualizar el comportamiento global de la demanda eléctrica diaria.
- Analizar la tendencia de largo plazo mediante media móvil.
- Comparar la evolución de la serie por años.
- Evaluar cambios en media y variabilidad mediante estadísticas móviles.
- Generar figuras y conclusiones útiles para el Capítulo 4 del TFM.

**BLOQUE 2 — Librerías y configuración**

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

**BLOQUE 3 — Rutas**


In [ ]:
# ==========================================
# BLOQUE 3 — Configuración de rutas
# ==========================================

from pathlib import Path

# Detectar si está dentro de notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUTS_FIGURES = PROJECT_ROOT / "outputs" / "figuras"
OUTPUTS_TABLES = PROJECT_ROOT / "outputs" / "tablas"

# Crear carpetas si no existen
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
OUTPUTS_FIGURES.mkdir(parents=True, exist_ok=True)
OUTPUTS_TABLES.mkdir(parents=True, exist_ok=True)

# Archivo de entrada
INPUT_FILE = DATA_PROCESSED / "data_daily_ready.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Archivo de entrada:", INPUT_FILE)

# Validación
if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"No se encontró el archivo: {INPUT_FILE}\n"
        "Verifica que el archivo exista en data/processed/"
    )

**BLOQUE 4 — Carga del dataset**

In [ ]:
df = pd.read_csv(INPUT_FILE, parse_dates=["date"])

print("Dimensión del dataset:", df.shape)
display(df.head())

**BLOQUE 5 — Preparación mínima de la serie**

In [ ]:
DATE_COL = "date"
Y_COL = "target_kwh"

if DATE_COL not in df.columns:
    raise ValueError(f"No se encontró la columna '{DATE_COL}'.")

if Y_COL not in df.columns:
    raise ValueError(f"No se encontró la columna '{Y_COL}'.")

df = df.sort_values(DATE_COL).set_index(DATE_COL)

y = df[[Y_COL]].copy()

print("Información de la serie objetivo:")
display(y.info())
display(y.head())
display(y.tail())

print("Frecuencia inferida:", pd.infer_freq(y.index))

**BLOQUE 6 — Estadísticos descriptivos**

In [ ]:
desc = y[Y_COL].describe().to_frame(name="value")
display(desc)

desc.to_csv(OUTPUTS_TABLES / "table_descriptive_statistics_target_kwh.csv")

**BLOQUE 7 — Serie temporal global**

In [ ]:
ts_plot = y.copy()

fig, ax = plt.subplots(figsize=(15, 4.8))

ax.plot(
    ts_plot.index,
    ts_plot[Y_COL],
    linewidth=0.9
)

ax.set_title("Serie temporal de la demanda eléctrica diaria (2022–2025)", fontsize=13)
ax.set_xlabel("Fecha", fontsize=11)
ax.set_ylabel("Demanda eléctrica diaria (kWh)", fontsize=11)

ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.set_xlim(ts_plot.index.min(), ts_plot.index.max())

ax.grid(True, alpha=0.25)
ax.tick_params(axis="x", rotation=0, labelsize=10)
ax.tick_params(axis="y", labelsize=10)

plt.tight_layout()
plt.savefig(OUTPUTS_FIGURES / "serie_temporal_demanda_diaria_2022_2025.png", dpi=300, bbox_inches="tight")
plt.show()

**BLOQUE 8 — Serie temporal con media móvil**

In [ ]:
ts_trend = y.copy()
ts_trend["ma_30"] = ts_trend[Y_COL].rolling(window=30, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(15, 4.8))

ax.plot(
    ts_trend.index,
    ts_trend[Y_COL],
    linewidth=0.8,
    alpha=0.65,
    label="Demanda diaria"
)

ax.plot(
    ts_trend.index,
    ts_trend["ma_30"],
    linewidth=2.0,
    label="Media móvil 30 días"
)

ax.set_title("Serie temporal de la demanda eléctrica diaria (2022–2025)", fontsize=13)
ax.set_xlabel("Fecha", fontsize=11)
ax.set_ylabel("Demanda eléctrica diaria (kWh)", fontsize=11)

ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
ax.set_xlim(ts_trend.index.min(), ts_trend.index.max())

ax.grid(True, alpha=0.25)
ax.legend(frameon=False, fontsize=10)
ax.tick_params(axis="x", rotation=0, labelsize=10)
ax.tick_params(axis="y", labelsize=10)

plt.tight_layout()
plt.savefig(OUTPUTS_FIGURES / "serie_temporal_demanda_diaria_tendencia.png", dpi=300, bbox_inches="tight")
plt.show()

**BLOQUE 9 — Visualización por años**

In [ ]:
years = sorted(y.index.year.unique())

fig, axes = plt.subplots(
    nrows=len(years),
    ncols=1,
    figsize=(15, 3 * len(years)),
    sharey=True
)

if len(years) == 1:
    axes = [axes]

for ax, year in zip(axes, years):
    yearly_data = y[y.index.year == year]

    ax.plot(yearly_data.index, yearly_data[Y_COL], linewidth=1)
    ax.set_title(f"Demanda eléctrica diaria — Año {year}")
    ax.set_ylabel("kWh")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Fecha")
plt.tight_layout()
plt.savefig(OUTPUTS_FIGURES / "demanda_diaria_por_anio.png", dpi=300, bbox_inches="tight")
plt.show()

**BLOQUE 10 — Medias móviles y desviación estándar móviles**

In [ ]:
window = 30

rolling_mean = y[Y_COL].rolling(window=window).mean()
rolling_std = y[Y_COL].rolling(window=window).std()

fig, ax = plt.subplots(figsize=(15, 5.5))

ax.plot(y.index, y[Y_COL], color="steelblue", alpha=0.35, label="Serie original")
ax.plot(rolling_mean.index, rolling_mean, color="red", linewidth=2, label="Media móvil (30 días)")
ax.plot(rolling_std.index, rolling_std, color="black", linewidth=2, label="Desv. estándar móvil (30 días)")

ax.set_title("Análisis rolling — media y desviación estándar móviles", fontsize=13)
ax.set_xlabel("Fecha", fontsize=11)
ax.set_ylabel("Demanda diaria (kWh)", fontsize=11)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUTS_FIGURES / "rolling_mean_std_demanda.png", dpi=300, bbox_inches="tight")
plt.show()

**BLOQUE 11 — Tabla resumen de picos más altos**

In [ ]:
top_peaks = (
    y.reset_index()
     .sort_values(Y_COL, ascending=False)
     .head(20)
     .reset_index(drop=True)
)

display(top_peaks)

top_peaks.to_csv(OUTPUTS_TABLES / "table_top20_demand_peaks.csv", index=False)

## Conclusiones del análisis exploratorio

A partir del análisis exploratorio de la serie de demanda eléctrica diaria se identifican los siguientes hallazgos principales:

1. La serie presenta una tendencia creciente de largo plazo, observable mediante la media móvil de 30 días, con un incremento progresivo del nivel medio de demanda desde 2022 hasta 2025.

2. Se observan picos de demanda aislados a lo largo de todo el periodo de estudio, con mayor frecuencia e intensidad en los años más recientes, lo que sugiere una mayor variabilidad en la serie.

3. La comparación por años muestra que, aunque existe una estructura general relativamente consistente, el nivel medio de la demanda y la dispersión aumentan a partir de 2023.

4. El análisis rolling confirma que ni la media ni la variabilidad permanecen constantes en el tiempo, lo cual constituye evidencia descriptiva de no estacionariedad en media y posible heterocedasticidad leve.

Estos resultados justifican la necesidad de profundizar posteriormente en el análisis formal de series temporales, incluyendo pruebas de estacionariedad, diferenciación y análisis de autocorrelación.